# Pyama results

Reads `analysis/` written by `analyze.ipynb` and writes user-facing `results/<sample>/` packs (XLSX tables + single-panel PNGs). Re-run this notebook after a plot-script change without repeating timeseries/AUC/fit.

This notebook owns sample names and merges them into `workspace/assay.json` as `samples[]`. It does not invent a Config signal channel: that comes from analyze (or from `analysis/PosN/ch*.csv` if assay.json has no channels yet). If `analysis.channels.signal` is missing, setup writes it as a one-element list from that resolved channel. Per-sample signal maps are not set here. Run `analyze.ipynb` first.


## Config

In [ ]:
from pathlib import Path

# Folder with analysis/ (from analyze.ipynb); results/<sample>/ is written here. Not the ND2/CZI file.
WORKSPACE = Path(r"Z:\\projects\LNPbinder\Experiments\20260731\Auswertung")

# Minutes between acquired frames (must match analyze.ipynb). 10.0 → t = 0, 10, 20, …
# Used for plot time axes. Interval in assay.json is written by analyze.ipynb, not here.
INTERVAL_MINUTES = 10.0

# One entry per sample. name is the folder under results/ (use filesystem-safe names).
# positions: zero-based field indices, same as roi/Pos{n} and analyze.
#   list(range(0, 40)) is positions 0 through 39 (Python half-open range), not 0..40 inclusive.
SAMPLES = [
    {"name": "A431_aiLNP_incubated", "positions": list(range(0, 40))},
    {"name": "A549_aiLNP_incubated", "positions": list(range(40, 80))},
    {"name": "A549_aiLNP", "positions": list(range(80, 121))},
    {"name": "A431_aiLNP", "positions": list(range(121, 159))},
]


## Setup

In [ ]:
from pyama.core import (
    merge_results_assay_json,
    require_analysis_dir,
    resolve_signal_channels,
    samples_to_mapping,
    slide_channel_labels,
)
from pyama.services import plot_auc, plot_fit, plot_timeseries

workspace = WORKSPACE.expanduser().resolve()
if not workspace.is_dir():
    raise FileNotFoundError(f"Workspace not found: {workspace}")
require_analysis_dir(workspace)
signal_channels = resolve_signal_channels(workspace)
mapping = samples_to_mapping(SAMPLES, signal_channel=signal_channels[0])
labels = slide_channel_labels(mapping)
assay_path = merge_results_assay_json(
    workspace, samples=SAMPLES, signal_channels=signal_channels
)
print(f"Workspace: {workspace}")
print(f"Signal channels: {signal_channels}")
print(f"Merged assay.json samples: {assay_path}")


## Plot timeseries

In [ ]:
for path in plot_timeseries.run_plot_timeseries(
    workspace=workspace,
    interval=INTERVAL_MINUTES,
    mapping=mapping,
    slide_channel_names=labels,
):
    print(plot_timeseries.format_written_timeseries_plot_message(path))


## Plot AUC

In [ ]:
for message in plot_auc.format_written_auc_plot_messages(
    plot_auc.run_plot_auc(workspace=workspace, mapping=mapping, slide_channel_names=labels)
):
    print(message)


## Plot fit

In [ ]:
for message in plot_fit.format_written_fit_plot_messages(
    plot_fit.run_plot_fit(
        workspace=workspace,
        interval=INTERVAL_MINUTES,
        mapping=mapping,
        slide_channel_names=labels,
    )
):
    print(message)

print("Results finished.")
